# PowerNext-AI | Screening Round Challenge
## The Black-Box Test Bench Challenge: Condition Monitoring & Hot-Spot Prediction
**Organized by CPRI with institutional partner MIT Bengaluru**
**Team Name:** PARS (PowerNext Analytical & Reliability System)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import HuberRegressor
from sklearn.ensemble import GradientBoostingRegressor, ExtraTreesRegressor
from lightgbm import LGBMRegressor

print("Libraries successfully imported!")

### 1. Load Datasets

In [ ]:
train_df = pd.read_csv("../dataset/training_data.csv")
test_df = pd.read_csv("../dataset/test_data.csv")
print(f"Training Data: {train_df.shape}")
print(f"Test Data: {test_df.shape}")
train_df.head()

### 2. Task 01: Physics-Grounded Anomaly Detection
Terminal sensors S1, S2, and S3 are coupled with Applied Voltage and Load Current. Residual deviations beyond 1.25 deg C or missing sensor readings represent hardware/recording faults rather than genuine operational regimes.

In [ ]:
valid_train = train_df[train_df["Validity_Label"] == "Valid"].copy()
X_vi = valid_train[["Applied_Voltage_kV", "Load_Current_A"]].values

sensor_models = {}
for sensor in ["Sensor_S1", "Sensor_S2", "Sensor_S3"]:
    hr = HuberRegressor(epsilon=1.35).fit(X_vi, valid_train[sensor].values)
    sensor_models[sensor] = hr

def evaluate_anomalies(df):
    n = len(df)
    labels = ["Valid"] * n
    scores = np.zeros(n)
    reasons = []
    all_cols = ["Applied_Voltage_kV", "Load_Current_A", "Ambient_Temperature_C", "Test_Duration_min", "Sensor_S1", "Sensor_S2", "Sensor_S3", "Sensor_S4"]
    dup_mask = df.duplicated(subset=all_cols, keep=False).values
    X_test_vi = df[["Applied_Voltage_kV", "Load_Current_A"]].values
    for i in range(n):
        r = df.iloc[i]
        r_reasons = []
        max_res = 0.0
        nulls = [s for s in ["Sensor_S1", "Sensor_S2", "Sensor_S3"] if pd.isna(r[s])]
        if nulls:
            r_reasons.append(f"Missing readings: {', '.join(nulls)}")
            max_res = max(max_res, 10.0)
        for sensor in ["Sensor_S1", "Sensor_S2", "Sensor_S3"]:
            if not pd.isna(r[sensor]):
                pred_s = sensor_models[sensor].predict(X_test_vi[i:i+1])[0]
                res = abs(r[sensor] - pred_s)
                max_res = max(max_res, res)
                if res > 1.25:
                    r_reasons.append(f"{sensor} physical deviation: {res:.2f}C")
        if dup_mask[i]:
            r_reasons.append("Duplicate operational setpoint")
            max_res = max(max_res, 5.0)
        scores[i] = max_res
        if r_reasons:
            labels[i] = "Invalid"
            reasons.append("; ".join(r_reasons))
        else:
            labels[i] = "Valid"
            reasons.append("Normal Operating Condition")
    return np.array(labels), scores, reasons

test_validity, test_anomaly_scores, test_reasons = evaluate_anomalies(test_df)
print(f"Test Invalid Records: {(test_validity == 'Invalid').sum()} / {len(test_df)}")

### 3. Task 02: Reference Parameter Estimation
Predict the critical hot-spot temperature rise using an ensemble of LightGBM, Gradient Boosting, and ExtraTrees.

In [ ]:
features = ["Applied_Voltage_kV", "Load_Current_A", "Ambient_Temperature_C", "Test_Duration_min", "Sensor_S1", "Sensor_S2", "Sensor_S3", "Sensor_S4"]
meds = valid_train[features].median()

def engineer_features(df):
    X = df[features].copy().fillna(meds)
    X["Power_Proxy_kVA"] = (X["Applied_Voltage_kV"] * X["Load_Current_A"]) / 1000.0
    X["Terminal_TempRise_Mean"] = (X["Sensor_S1"] + X["Sensor_S2"]) / 2.0
    X["Temp_Gradient_S3_S1"] = X["Sensor_S3"] - X["Sensor_S1"]
    X["Temp_Gradient_S2_S1"] = X["Sensor_S2"] - X["Sensor_S1"]
    X["Ambient_Plus_Terminal"] = X["Ambient_Temperature_C"] + X["Terminal_TempRise_Mean"]
    return X

X_train_fe = engineer_features(valid_train)
y_train = valid_train["Reference_Parameter"].values
X_test_fe = engineer_features(test_df)

lgb = LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=31, subsample=0.85, random_state=42, verbose=-1).fit(X_train_fe, y_train)
gbr = GradientBoostingRegressor(n_estimators=250, learning_rate=0.04, max_depth=4, subsample=0.85, random_state=42).fit(X_train_fe, y_train)
etr = ExtraTreesRegressor(n_estimators=200, max_depth=12, random_state=42).fit(X_train_fe, y_train)

preds = np.round(0.50 * lgb.predict(X_test_fe) + 0.30 * gbr.predict(X_test_fe) + 0.20 * etr.predict(X_test_fe), 4)
print(f"Predicted Reference Parameter: Min={preds.min()}, Max={preds.max()}")

### 4. Task 03: Automated Summary & Submission Output

In [ ]:
submission_df = pd.DataFrame({
    "Test_ID": test_df["Test_ID"],
    "Predicted_Reference_Parameter": preds,
    "Validity_Label": test_validity
})
submission_df.to_csv("../deliverables/PARS.csv", index=False)
print("Exported ../deliverables/PARS.csv successfully!")

sorted_idx = np.argsort(-test_anomaly_scores)[:3]
print("Top 3 Test IDs requiring attention:", list(test_df.iloc[sorted_idx]["Test_ID"]))
